# Methods figures
-------

1. **Reference labels** — the labelled parcels, old-growth (light green) versus
   non-old-growth (orange); unlabelled parcels are omitted.
2. **Spatial folds** — the labelled parcels coloured by the six-fold cross-validation
   partition of notebook 008 (which balances total parcel area and old-growth area
   jointly across folds).
3. **Predicted probability** — the published old-growth probability raster
   (`results/final/ogf_probability_3035_10m.tif`), classified into four 25-percentage-point
   classes.
4. **Parcel predictions** — the per-parcel old-growth probability
   (`results/final/ogf_labels_predictions.gpkg`), binned into the same four
   25-percentage-point classes.
5. **TESSERA embedding** — a false-colour view of the TESSERA 2020 embedding over the AOI.
6. **AlphaEarth embedding** — the same false-colour view for the AlphaEarth 2020 embedding.

In [ ]:
NOTEBOOK = "012_methods_figures"

import contextily as ctx
import geopandas as gpd
import matplotlib.patheffects as path_effects
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from matplotlib.colors import ListedColormap, to_rgba
from matplotlib.patches import Polygon, Rectangle
from rasterio.enums import Resampling
from rasterio.features import rasterize
from rasterio.transform import from_bounds
from rasterio.warp import reproject
from scipy.ndimage import binary_erosion, gaussian_filter

from utils.paths import get_project_paths
from utils.style import save_figure, use_publication_style
from utils.terminology import DISPLAY_CRS, FOLD_COLOURS, PALETTE_CATEGORICAL, SEMANTIC_COLOURS

use_publication_style()

paths = get_project_paths()
# Web-tile cache on disk (contextily's default is a temp dir that is lost between sessions),
# so the location inset and the no-orthomosaic fallback need the network only once.
ctx.set_cache_dir(paths.cache / "tiles")
ORTHO = paths.raw / "rasters" / "orthophotoplan" / "mosaic_rgb_vis.tif"
PROB = paths.results / "final" / "ogf_probability_3035_10m.tif"
TESSERA = paths.processed / "rasters" / "tessera_10m" / "tessera_2020_3035_10m.tif"
ALPHAEARTH = paths.processed / "rasters" / "alphaearth_10m" / "alphaearth_2020_3035_10m.tif"
AOI = paths.aoi
LABELS = paths.labels / "ogf_reference_labels_partitioned.gpkg"
PRED_GPKG = paths.results / "final" / "ogf_labels_predictions.gpkg"
LABELS_LAYER = "ogf_reference_labels_partitioned"
PRED_LAYER = "ogf_labels_predictions"

# Old-growth probability classes, low to high, from the shared categorical palette.
PROB_BIN_COLOURS = tuple(
    PALETTE_CATEGORICAL[name] for name in ("magenta", "orange", "light_green", "teal")
)

### Figure layout constants

In [ ]:
# Figure geometry. The page is the figures' final publication size. The basemap is
# rendered at 600 dpi and the PDF written at the journal photo standard of 300 dpi with
# its rasters JPEG-encoded at quality 85 (the line work is vector regardless); path
# coordinates are rounded to 0.01 pt, which shrinks the parcel maps by a third at no
# visible cost.
FIG_W_MM = 100.0  # publication width of figures 1-4 (height follows: 100 / 1.6 = 62.5 mm)
RENDER_W = 2362  # basemap pixels = 100 mm at 600 dpi
RENDER_DPI = 600
PDF_DPI = 300
PDF_JPEG_QUALITY = 85
PATH_DECIMALS = 2
FILL_ALPHA = 1.0  # opaque overlays (no transparency)
RASTER_ALPHA = 1.0  # opaque probability-raster overlay
BASEMAP_ALPHA = 0.9  # faint orthomosaic so the opaque overlays read with more contrast
EDGE_LW = 0.12  # thin black polygon borders

# Legend (millimetres); bottom-left, its bottom flush with the mosaic bottom
LEG_LEFT = 2.4
LEG_BOTTOM = 2.6
SW_W, SW_H = 6.0, 3.0  # swatch width/height
ROW_PITCH = 4.0  # vertical key-to-key pitch
TEXT_GAP = 1.5
LEG_FONT = 6.0
LEG_TITLE_GAP = 1.4  # gap between the legend body and its title

# Scale bar (four alternating segments), bottom-right; the right edge is fixed by
# SCALE_RIGHT_MARGIN, so widening SCALE_KM extends the bar leftwards only.
SCALE_KM = 20.0
SCALE_SEGMENTS = 4
SCALE_RIGHT_MARGIN = 2.4
SCALE_BAR_Y = 2.7
SCALE_BAR_H = 1.3
NORTH_FRAC = 0.06  # north-arrow size as a fraction of the shorter map dimension

## Basemap

Read the orthomosaic once, downsampled to the render width via its overviews. The mosaic
is the only rasterised element; its nodata (all-zero) pixels become transparent so the
blank margins of the page show through.

When the orthomosaic is not available (it is not redistributed), OpenStreetMap tiles are
fetched instead and warped onto a metric grid in the display CRS covering the AOI plus a
margin, at the same page size and aspect, so the scale bar and north arrow stay exact.

In [ ]:
ASPECT = 1.6  # publication aspect of figures 1-4 (100 x 62.5 mm), set by the orthomosaic

if ORTHO.exists():
    BASEMAP_SOURCE = "orthomosaic"
    with rasterio.open(ORTHO) as src:
        aspect = src.width / src.height
        H = int(round(RENDER_W / aspect))
        base = src.read(out_shape=(3, H, RENDER_W), resampling=Resampling.average)
        bounds = src.bounds
        ORTHO_CRS = src.crs
    # Clean data edge: average-resampling blends nodata (0) into boundary pixels, leaving a
    # dark ragged ring. Drop near-black pixels, erode the blended ring, then feather the
    # alpha for a smooth anti-aliased boundary. The whole basemap is then scaled by
    # BASEMAP_ALPHA so it reads faintly under the opaque overlays.
    lum = base.astype("float32").sum(axis=0)
    data_mask = binary_erosion(lum > 30.0, iterations=1)
    alpha = np.clip(gaussian_filter(data_mask.astype("float32"), sigma=0.7), 0.0, 1.0)
    alpha = alpha * BASEMAP_ALPHA
    print(f"orthomosaic {src.width}x{src.height} -> render {RENDER_W}x{H}")
else:
    # Fallback: the orthomosaic is not redistributed, so the panels are drawn over
    # OpenStreetMap tiles (map data © OpenStreetMap contributors, ODbL). The tiles are
    # fetched in Web Mercator and warped onto a metric grid in the display CRS (UTM 35N)
    # that covers the AOI plus a margin and is extended to the publication aspect, so the
    # scale bar and north arrow stay exact. Needs network access; contextily caches tiles.
    import contextily as ctx
    import rasterio.crs
    from rasterio.coords import BoundingBox
    from rasterio.warp import transform_bounds

    BASEMAP_SOURCE = "OpenStreetMap"
    BASEMAP_PROVIDER = ctx.providers.OpenStreetMap.Mapnik  # OpenTopoMap (the inset's) also works
    # The OpenStreetMap tile policy requires an identifying User-Agent; anonymous
    # requests get "access blocked" placeholder tiles.
    BASEMAP_HEADERS = {
        "User-Agent": "old-growth-forests-publication notebook 012 (contact: trr26@cam.ac.uk)"
    }
    BASEMAP_MARGIN_M = 1_500.0
    aspect = ASPECT
    H = int(round(RENDER_W / aspect))
    minx, miny, maxx, maxy = gpd.read_file(AOI).to_crs(DISPLAY_CRS).total_bounds
    minx, miny = minx - BASEMAP_MARGIN_M, miny - BASEMAP_MARGIN_M
    maxx, maxy = maxx + BASEMAP_MARGIN_M, maxy + BASEMAP_MARGIN_M
    width, height = maxx - minx, maxy - miny
    if width / height < aspect:  # too tall for the page: widen symmetrically
        extra = height * aspect - width
        minx, maxx = minx - extra / 2, maxx + extra / 2
    else:  # too wide: heighten symmetrically
        extra = width / aspect - height
        miny, maxy = miny - extra / 2, maxy + extra / 2
    bounds = BoundingBox(minx, miny, maxx, maxy)
    ORTHO_CRS = rasterio.crs.CRS.from_user_input(DISPLAY_CRS)
    west, south, east, north = transform_bounds(DISPLAY_CRS, "EPSG:3857", minx, miny, maxx, maxy)
    # Tile zoom that roughly matches the render resolution (Web Mercator: 156,543 m per
    # pixel at zoom 0), capped at 13 to keep the tile request modest.
    zoom = min(13, int(np.ceil(np.log2(156_543.03 * RENDER_W / (east - west)))))
    tiles, (t_west, t_east, t_south, t_north) = ctx.bounds2img(
        west,
        south,
        east,
        north,
        zoom=zoom,
        source=BASEMAP_PROVIDER,
        headers=BASEMAP_HEADERS,
        ll=False,
    )
    tile_transform = from_bounds(t_west, t_south, t_east, t_north, tiles.shape[1], tiles.shape[0])
    base = np.zeros((3, H, RENDER_W), dtype="uint8")
    for band in range(3):
        reproject(
            source=np.ascontiguousarray(tiles[:, :, band]),
            destination=base[band],
            src_transform=tile_transform,
            src_crs="EPSG:3857",
            dst_transform=from_bounds(minx, miny, maxx, maxy, RENDER_W, H),
            dst_crs=ORTHO_CRS,
            resampling=Resampling.bilinear,
        )
    alpha = np.full((H, RENDER_W), BASEMAP_ALPHA, dtype="float32")
    print(f"no orthomosaic at {ORTHO}; OpenStreetMap tiles at zoom {zoom} -> render {RENDER_W}x{H}")

FIG_H_MM = FIG_W_MM / aspect
FIGSIZE = (FIG_W_MM / 25.4, FIG_H_MM / 25.4)
EXTENT = [bounds.left, bounds.right, bounds.bottom, bounds.top]
rgba_base = np.dstack([np.transpose(base, (1, 2, 0)), (alpha * 255).astype("uint8")])
print(f"page {FIG_W_MM:.0f} x {FIG_H_MM:.1f} mm (aspect {aspect:.2f}); basemap: {BASEMAP_SOURCE}")

## Helpers

In [ ]:
def warp_prob():
    """Warp the EPSG:3035 probability raster onto the orthomosaic grid (masked)."""
    with rasterio.open(PROB) as src:
        dst = np.full((H, RENDER_W), -9999.0, dtype="float32")
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=from_bounds(
                bounds.left, bounds.bottom, bounds.right, bounds.top, RENDER_W, H
            ),
            dst_crs=ORTHO_CRS,
            resampling=Resampling.bilinear,
            src_nodata=-9999.0,
            dst_nodata=-9999.0,
        )
    return np.ma.masked_equal(dst, -9999.0)


def prob_bins(n):
    """Equal-width probability bins shared by figures 3 and 4.

    Returns the interior edges for ``np.digitize``, the percentage-range key labels and
    the palette colour of each class. ``n`` must match ``PROB_BIN_COLOURS``, so
    ``prob_bins(4)`` gives the published figures their four 25-percentage-point classes.
    """
    if n != len(PROB_BIN_COLOURS):
        raise ValueError(f"prob_bins expects n={len(PROB_BIN_COLOURS)}, got {n}")
    edges = [i / n for i in range(1, n)]
    labels = [f"{100 * i // n}–{100 * (i + 1) // n}%" for i in range(n)]  # noqa: RUF001
    return edges, labels, list(PROB_BIN_COLOURS)


def new_figure():
    """Transparent figure: a map axes (metre coords) and an overlay axes (mm coords)."""
    fig = plt.figure(figsize=FIGSIZE, facecolor="none")
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(rgba_base, extent=EXTENT, origin="upper", interpolation="nearest", zorder=0)
    ax.set_xlim(bounds.left, bounds.right)
    ax.set_ylim(bounds.bottom, bounds.top)
    ax.set_aspect("auto")
    ax.set_axis_off()
    ax.set_facecolor("none")
    leg = fig.add_axes([0, 0, 1, 1], zorder=5)
    leg.set_xlim(0, FIG_W_MM)
    leg.set_ylim(0, FIG_H_MM)
    leg.set_aspect("auto")
    leg.set_axis_off()
    leg.set_facecolor("none")
    if BASEMAP_SOURCE != "orthomosaic":
        leg.text(
            FIG_W_MM - 1.5,
            FIG_H_MM - 1.2,
            "Map data © OpenStreetMap contributors (ODbL)",
            ha="right",
            va="top",
            fontsize=4.5,
            color="black",
            path_effects=[path_effects.withStroke(linewidth=1.2, foreground="white")],
            zorder=7,
        )
    return fig, ax, leg


def legend_title(leg, top_mm, title):
    """Draw a bold legend title with its baseline ``LEG_TITLE_GAP`` above ``top_mm``."""
    leg.text(
        LEG_LEFT,
        top_mm + LEG_TITLE_GAP,
        title,
        va="bottom",
        ha="left",
        fontsize=LEG_FONT + 0.5,
        fontweight="bold",
        color="black",
        zorder=7,
    )


def swatch_legend(leg, entries, title=None, bottom=LEG_BOTTOM):
    """Bottom-aligned key legend; ``entries`` are (colour, label) top-to-bottom.

    ``bottom`` is the y (mm) of the lowest swatch, so a figure can nudge the whole legend
    down or up without touching the shared default.
    """
    n = len(entries)
    for i, (col, label) in enumerate(entries):
        y = bottom + (n - 1 - i) * ROW_PITCH
        leg.add_patch(
            Rectangle(
                (LEG_LEFT, y),
                SW_W,
                SW_H,
                facecolor=to_rgba(col, FILL_ALPHA),
                edgecolor="black",
                linewidth=0.4,
                clip_on=False,
                zorder=6,
            )
        )
        leg.text(
            LEG_LEFT + SW_W + TEXT_GAP,
            y + SW_H / 2,
            label,
            va="center",
            ha="left",
            fontsize=LEG_FONT,
            color="black",
            zorder=6,
        )
    if title:
        legend_title(leg, bottom + (n - 1) * ROW_PITCH + SW_H, title)


def add_scale_bar(leg):
    """Alternating black/white scale bar (SCALE_KM long) in the bottom-right (mm coords)."""
    total_mm = SCALE_KM * 1000.0 / (bounds.right - bounds.left) * FIG_W_MM
    seg = total_mm / SCALE_SEGMENTS
    x_left = FIG_W_MM - SCALE_RIGHT_MARGIN - total_mm
    for k in range(SCALE_SEGMENTS):
        leg.add_patch(
            Rectangle(
                (x_left + k * seg, SCALE_BAR_Y),
                seg,
                SCALE_BAR_H,
                facecolor="black" if k % 2 == 0 else "white",
                edgecolor="black",
                linewidth=0.4,
                clip_on=False,
                zorder=7,
            )
        )
    leg.text(
        x_left + total_mm / 2,
        SCALE_BAR_Y + SCALE_BAR_H + 0.5,
        f"{SCALE_KM:.0f} km",
        ha="center",
        va="bottom",
        fontsize=LEG_FONT,
        color="black",
        path_effects=[path_effects.withStroke(linewidth=1.4, foreground="white")],
        zorder=7,
    )


def add_north_arrow(ax):
    """Compass needle (north up) in the top-left corner (metre coords), as in 016."""
    x0, x1 = sorted(ax.get_xlim())
    y0, y1 = sorted(ax.get_ylim())
    width, height = x1 - x0, y1 - y0
    size = NORTH_FRAC * min(width, height)
    cx, cy = x0 + 0.07 * width, y1 - 0.11 * height
    half = 0.36 * size
    pts = [
        (cx, cy + 0.40 * size),
        (cx + half, cy - 0.40 * size),
        (cx, cy - 0.16 * size),
        (cx - half, cy - 0.40 * size),
    ]
    ax.add_patch(
        Polygon(
            pts,
            closed=True,
            facecolor="black",
            edgecolor="white",
            linewidth=0.5,
            clip_on=False,
            zorder=8,
        )
    )
    ax.text(
        cx,
        cy + 0.40 * size + 0.004 * height,
        "N",
        ha="center",
        va="bottom",
        fontsize=LEG_FONT,
        color="black",
        path_effects=[path_effects.withStroke(linewidth=1.4, foreground="white")],
        zorder=8,
    )


def decorate(fig, ax, leg):
    """Scale bar and north arrow shared by every figure."""
    add_scale_bar(leg)
    add_north_arrow(ax)


def save_map(fig, name, dpi=PDF_DPI):
    """Write the publication PDF: rasters at ``dpi`` / quality 85, coordinates at 0.01 pt."""
    save_figure(fig, name, dpi=dpi, raster_quality=PDF_JPEG_QUALITY, path_decimals=PATH_DECIMALS)

## Study-area location

In [ ]:
# Location inset. Locates the Făgăraș Mountains AOI within Romania and the Carpathian
# mountain range (the area-of-applicability extension), after the archive poster inset
# (018_create_fagaras_location_inset.py in the original research repository), with two
# changes: a topographic web basemap (Esri World Topographic Map: shaded relief, borders,
# place names) carries the background instead of flat land/sea fills, and Romania is an
# unfilled outline. The country outline is the national boundary from the OSM extract
# already downloaded in notebook 001, so the inset needs no dataset of its own; the basemap
# tiles are fetched from the web on first run and cached by contextily.
import math
import textwrap

# Web Mercator, the tile CRS. The map is drawn in it so the tiles are placed
# without reprojection: warping them into another CRS (e.g. the project's EPSG:3035) rotates
# the raster and slants the tiles' baked-in place-name labels. This is a locator inset, not
# an analysis product, so the tile-native projection is fine.
MAP_CRS = "EPSG:3857"
TILE_ZOOM = 8  # tile detail level; the raster is resampled to the figure dpi on save
INSET_PAD_IN = 0.02  # white space left around the tight crop, in inches
# The inset is placed 100 mm wide in the publication. Its page is wider than that (the
# tight crop keeps the coordinate labels), so save_inset writes the tiles at the dpi that
# lands at exactly PDF_DPI once the page is scaled down to the printed width.
INSET_PUB_W_MM = 100.0
ATTRIBUTION_FONT_SIZE = 6  # credit line; deliberately below the 14 pt place labels
# Characters per attribution line, held short enough that the block clears the scale bar in
# the bottom-right. Measured in Charis SIL across the 315.6 pt axes, an 89-character line
# fills 84% of the width at size 6 (0.94% per character); the scale bar starts at about 85%,
# so lines are capped at 80 characters (75%). The Esri credit wraps to three lines.
ATTRIBUTION_WRAP = 80
# Esri's World Topographic Map is served from a CDN that resolves and answers reliably from
# the university network, where the community OSM/OpenTopoMap tile hosts intermittently
# time out; the tiles are cached on disk after the first run (see the setup cell).
BASEMAP = ctx.providers.Esri.WorldTopoMap
ATTRIBUTION = BASEMAP["attribution"]  # Esri's required data-source credit

ROMANIA_EDGE = "#3d4f38"
AOI_FILL = "#d4e2c8"
SITE_LINE = "black"  # AOI outline and label arrows
TEXT_COLOR = "#2d312c"
SITE_TEXT_COLOR = "#4b241d"
INSET_SCALE_FONT = 9.0  # scale-bar label; larger than the map-figure LEG_FONT

OSM_GPKG = paths.raw / "vectors" / "open_street_map" / "romania.gpkg"


def save_inset(fig, name):
    """Write the tight-cropped inset PDF with its tiles at ``PDF_DPI`` on the printed page.

    The crop goes through ``save_figure`` rather than a second ``savefig`` over its
    output, which would drop the raster compression and the coordinate rounding.
    """
    crop_w_mm = (fig.get_tightbbox().width + 2 * INSET_PAD_IN) * 25.4
    dpi = round(PDF_DPI * INSET_PUB_W_MM / crop_w_mm)
    (written,) = save_figure(
        fig,
        name,
        dpi=dpi,
        raster_quality=PDF_JPEG_QUALITY,
        path_decimals=PATH_DECIMALS,
        bbox_inches="tight",
        pad_inches=INSET_PAD_IN,
    )
    print(
        f"[figure] {written.relative_to(paths.repo_root)} ({crop_w_mm:.1f} mm page at {dpi} dpi "
        f"-> {PDF_DPI} dpi at {INSET_PUB_W_MM:.0f} mm printed)"
    )
    return written


# Romania: the single national-level polygon in the Geofabrik OSM extract.
romania = gpd.read_file(
    OSM_GPKG, layer="gis_osm_adminareas_a_free", where="fclass = 'national'"
).to_crs(MAP_CRS)

aoi = gpd.read_file(paths.aoi).to_crs(MAP_CRS)
aoi_outline = gpd.GeoSeries([aoi.geometry.union_all()], crs=MAP_CRS)
aoi_centroid = aoi_outline.iloc[0].centroid

rb = romania.total_bounds
romania_w, romania_h = rb[2] - rb[0], rb[3] - rb[1]
halo = [path_effects.withStroke(linewidth=2.0, foreground="white")]
fig_height = 4.4

from matplotlib.ticker import FuncFormatter

km_fmt = FuncFormatter(lambda v, _pos: f"{v / 1000:,.0f}")

# Frame: the Carpathian mountain range and Romania together, recentred on Romania.
carpathians = gpd.read_file(
    paths.processed / "vectors" / "european_mountain_areas" / "carpathians_3035.gpkg"
).to_crs(MAP_CRS)
cb = carpathians.total_bounds
x0, y0 = min(cb[0], rb[0]), min(cb[1], rb[1])
x1, y1 = max(cb[2], rb[2]), max(cb[3], rb[3])
pad = 0.05 * max(x1 - x0, y1 - y0)
x0, y0, x1, y1 = x0 - pad, y0 - pad, x1 + pad, y1 + pad
# Recentre the frame on Romania: widen the extent symmetrically about the centre of
# Romania's bounds so the country sits in the middle while the Carpathians stay in view.
rc_x, rc_y = (rb[0] + rb[2]) / 2, (rb[1] + rb[3]) / 2
half_w = max(rc_x - x0, x1 - rc_x)
half_h = max(rc_y - y0, y1 - rc_y)
x0, x1 = rc_x - half_w, rc_x + half_w
y0, y1 = rc_y - half_h, rc_y + half_h

fig, ax = plt.subplots(figsize=(fig_height * (x1 - x0) / (y1 - y0), fig_height), dpi=300)
ax.set_xlim(x0, x1)
ax.set_ylim(y0, y1)
ax.set_aspect("equal", adjustable="box")
ctx.add_basemap(ax, crs=MAP_CRS, source=BASEMAP, zoom=TILE_ZOOM, attribution=False, zorder=0)
romania.plot(ax=ax, facecolor="none", edgecolor=ROMANIA_EDGE, linewidth=1.6, zorder=2)
aoi_outline.plot(ax=ax, color=AOI_FILL, edgecolor="none", zorder=4)
aoi_outline.boundary.plot(ax=ax, color=SITE_LINE, linewidth=0.9, zorder=5)

ax.text(
    aoi_centroid.x,
    aoi_centroid.y + 0.16 * romania_h,
    "Romania",
    ha="center",
    va="center",
    fontsize=14,
    color=TEXT_COLOR,
    path_effects=halo,
    zorder=6,
)
# Arrow and label drawn separately so the arrow tail sits exactly at the label's vertical
# mid-gap (va="center" puts the label centre there), just right of the text.
site_label_x = aoi_centroid.x - 0.28 * romania_w
site_label_y = aoi_centroid.y - 0.20 * romania_h
site_arrow = ax.annotate(
    "",
    xy=(aoi_centroid.x, aoi_centroid.y),
    xytext=(site_label_x, site_label_y),
    arrowprops={
        "arrowstyle": "->",
        "color": SITE_LINE,
        "linewidth": 1.4,
        "shrinkA": 2.0,
        "shrinkB": 7.0,
    },
    zorder=6,
)
site_arrow.arrow_patch.set_path_effects(
    [path_effects.withStroke(linewidth=3.0, foreground="white")]
)
site_label = ax.text(
    site_label_x - 0.008 * (x1 - x0),
    site_label_y,
    "Făgăraș\nMountains",
    fontsize=14,
    color=SITE_TEXT_COLOR,
    ha="right",
    va="center",
    ma="center",
    zorder=6,
)
site_label.set_path_effects(halo)

attribution = ctx.add_attribution(
    ax,
    "\n".join(textwrap.wrap(ATTRIBUTION, ATTRIBUTION_WRAP)),
    font_size=ATTRIBUTION_FONT_SIZE,
    zorder=7,
)
attribution.set_wrap(False)
attribution.set_path_effects([])
attribution.set_color("black")
attribution.set_verticalalignment("bottom")
attribution.set_bbox(
    {"facecolor": "white", "alpha": 0.7, "edgecolor": "none", "boxstyle": "square,pad=0.25"}
)

ax.xaxis.set_major_formatter(km_fmt)
ax.yaxis.set_major_formatter(km_fmt)
ax.tick_params(labelsize=7, length=2.5, pad=1.5)
ax.set_xlabel(f"Easting (km, {MAP_CRS})", fontsize=8, labelpad=2.0)
ax.set_ylabel(f"Northing (km, {MAP_CRS})", fontsize=8, labelpad=2.0)
for spine in ax.spines.values():
    spine.set_visible(False)

# Scale bar (the map-figure style, drawn inset-locally): Web Mercator inflates ground
# distance by 1/cos(latitude), so the bar spans the map-unit length of a true 200 km at
# the view's central latitude and reads correctly where it sits.
SCALE_KM_INSET = 200.0
centre_lat = math.degrees(2.0 * math.atan(math.exp(((y0 + y1) / 2) / 6378137.0)) - math.pi / 2)
bar_len = SCALE_KM_INSET * 1000.0 / math.cos(math.radians(centre_lat))
bar_seg = bar_len / 4
bar_h = 0.018 * (y1 - y0)
bar_x = x1 - 0.015 * (x1 - x0) - bar_len
bar_y = y1 - 0.055 * (y1 - y0)
for k in range(4):
    ax.add_patch(
        Rectangle(
            (bar_x + k * bar_seg, bar_y),
            bar_seg,
            bar_h,
            facecolor="black" if k % 2 == 0 else "white",
            edgecolor="black",
            linewidth=0.4,
            zorder=7,
        )
    )
ax.text(
    bar_x + bar_len / 2,
    bar_y + bar_h + 0.008 * (y1 - y0),
    f"{SCALE_KM_INSET:.0f} km",
    ha="center",
    va="bottom",
    fontsize=INSET_SCALE_FONT,
    color="black",
    path_effects=[path_effects.withStroke(linewidth=1.4, foreground="white")],
    zorder=7,
)
size_n = NORTH_FRAC * min(x1 - x0, y1 - y0)
cx_n, cy_n = x0 + 0.030 * (x1 - x0), y1 - 0.075 * (y1 - y0)
half_n = 0.36 * size_n
ax.add_patch(
    Polygon(
        [
            (cx_n, cy_n + 0.40 * size_n),
            (cx_n + half_n, cy_n - 0.40 * size_n),
            (cx_n, cy_n - 0.16 * size_n),
            (cx_n - half_n, cy_n - 0.40 * size_n),
        ],
        closed=True,
        facecolor="black",
        edgecolor="white",
        linewidth=0.5,
        clip_on=False,
        zorder=8,
    )
)
ax.text(
    cx_n,
    cy_n + 0.40 * size_n + 0.004 * (y1 - y0),
    "N",
    ha="center",
    va="bottom",
    fontsize=LEG_FONT,
    color="black",
    path_effects=[path_effects.withStroke(linewidth=1.4, foreground="white")],
    zorder=8,
)

save_inset(fig, f"{NOTEBOOK}/location_inset_romania_carpathians")
plt.show()

## Figure 1: reference labels

Labelled parcels only (`ogf` not null): old-growth in light green, non-old-growth in
orange, with thin black borders.

In [ ]:
lab = gpd.read_file(LABELS, layer=LABELS_LAYER)
lab = lab[lab["ogf"].notna()].copy()
lab["geometry"] = lab.geometry.simplify(8)
lab = lab.to_crs(ORTHO_CRS)

OGF_GREEN = PALETTE_CATEGORICAL["light_green"]
NON_OGF_ORANGE = SEMANTIC_COLOURS["non_ogf"]
fig, ax, leg = new_figure()
for val, col in [(1.0, OGF_GREEN), (0.0, NON_OGF_ORANGE)]:
    lab[lab["ogf"] == val].plot(
        ax=ax, facecolor=to_rgba(col, FILL_ALPHA), edgecolor="black", linewidth=EDGE_LW, zorder=2
    )
swatch_legend(
    leg,
    [(OGF_GREEN, "Old-growth"), (NON_OGF_ORANGE, "Non-old-growth")],
    title="Reference label",
)
decorate(fig, ax, leg)
save_map(fig, f"{NOTEBOOK}/fig_1_panel_a_reference_labels")
plt.show()

## Figure 2: spatial folds

The six cross-validation folds, each one of the six palette hues (the full palette gives
the most contrast available); thin black borders.

In [ ]:
fld = gpd.read_file(LABELS, layer=LABELS_LAYER)
fld = fld[fld["fold_id"].notna()].copy()
fld["fold_id"] = fld["fold_id"].astype(int)
fld["geometry"] = fld.geometry.simplify(8)
fld = fld.to_crs(ORTHO_CRS)

fig, ax, leg = new_figure()
entries = []
for f in range(1, 7):
    col = FOLD_COLOURS[f]
    fld[fld["fold_id"] == f].plot(
        ax=ax, facecolor=to_rgba(col, FILL_ALPHA), edgecolor="black", linewidth=EDGE_LW, zorder=2
    )
    entries.append((col, f"Fold {f}"))
swatch_legend(leg, entries)
decorate(fig, ax, leg)
save_map(fig, f"{NOTEBOOK}/fig_1_panel_b_spatial_folds")
plt.show()

## Figure 3: predicted probability

The published old-growth probability raster classified into the four shared 25-percentage-point classes, drawn as a flat colour per class (magenta, orange, light green and teal from the shared categorical palette) with the same key-style legend as figure 4.

In [ ]:
prob = warp_prob()

bin_edges, bin_labels, bin_cols = prob_bins(4)
prob_idx = np.ma.masked_array(
    np.digitize(prob.filled(-9999.0), bin_edges), mask=np.ma.getmaskarray(prob)
)

fig, ax, leg = new_figure()
ax.imshow(
    prob_idx,
    extent=EXTENT,
    origin="upper",
    cmap=ListedColormap(bin_cols),
    vmin=-0.5,
    vmax=len(bin_cols) - 0.5,
    alpha=RASTER_ALPHA,
    interpolation="nearest",
    zorder=2,
)
# Same legend nudge as figure 4 so the title clears the map.
swatch_legend(
    leg,
    list(zip(bin_cols, bin_labels, strict=True)),
    title="OGF probability",
    bottom=1.0,
)
decorate(fig, ax, leg)
save_map(fig, f"{NOTEBOOK}/fig_1_panel_e_predicted_probability")
plt.show()

## Figure 4: parcel predictions

Per-parcel old-growth probability (`OGF_probability`) binned into four equal
25-percentage-point classes, coloured magenta, orange, light green and teal from the
shared categorical palette; thin black borders.

In [ ]:
pred = gpd.read_file(PRED_GPKG, layer=PRED_LAYER)
pred = pred[pred["OGF_probability"].notna()].copy()
pred["geometry"] = pred.geometry.simplify(8)
pred = pred.to_crs(ORTHO_CRS)

bin_edges, bin_labels, bin_cols = prob_bins(4)
pred["bin"] = np.digitize(pred["OGF_probability"], bin_edges)

fig, ax, leg = new_figure()
for i in range(4):
    pred[pred["bin"] == i].plot(
        ax=ax,
        facecolor=to_rgba(bin_cols[i], FILL_ALPHA),
        edgecolor="black",
        linewidth=EDGE_LW,
        zorder=2,
    )
# Nudge the whole legend down a little so the title clears the map.
swatch_legend(
    leg,
    list(zip(bin_cols, bin_labels, strict=True)),
    title="OGF probability",
    bottom=1.0,
)
decorate(fig, ax, leg)
save_map(fig, f"{NOTEBOOK}/fig_1_panel_f_parcel_predictions")
plt.show()

## TESSERA embedding (false colour)

In [ ]:
# Shared false-colour embedding renderer for panels c.3 (AlphaEarth) and c.4 (TESSERA): take
# the first three channels, reproject to the display CRS (UTM 35N) so the scene reads
# upright, crop to and mask by the AOI, and map the channels to R/G/B with a per-channel
# 2-98 percentile stretch over the kept pixels. The page is the panels' final publication
# size at the same 600 dpi as the other figures, so the raster carries no more pixels than
# print needs; the height follows the AOI extent (~20.5 mm). No basemap, legend, scale bar
# or frame.
EMB_W_MM = 34.7  # publication width of the embedding panels
EMB_RENDER_W = 820  # raster pixels = 34.7 mm at 600 dpi


def embedding_figure(raster, name):
    aoi = gpd.read_file(AOI).to_crs(DISPLAY_CRS)
    t_left, t_bottom, t_right, t_top = aoi.total_bounds
    t_aspect = (t_right - t_left) / (t_top - t_bottom)
    th = int(round(EMB_RENDER_W / t_aspect))
    t_transform = from_bounds(t_left, t_bottom, t_right, t_top, EMB_RENDER_W, th)

    with rasterio.open(raster) as src:
        bands = [1, 2, 3]  # the first three embedding dimensions
        cube = np.full((3, th, EMB_RENDER_W), -9999.0, dtype="float32")
        for k, bi in enumerate(bands):
            reproject(
                source=rasterio.band(src, bi),
                destination=cube[k],
                src_transform=src.transform,
                src_crs=src.crs,
                dst_transform=t_transform,
                dst_crs=DISPLAY_CRS,
                src_nodata=-9999.0,
                dst_nodata=-9999.0,
                resampling=Resampling.bilinear,
            )

    # Valid pixels (nodata -9999 bleeds very negative under warping), masked to the AOI.
    valid = (cube > -100).all(axis=0)
    aoi_mask = rasterize(
        [(g, 1) for g in aoi.geometry],
        out_shape=(th, EMB_RENDER_W),
        transform=t_transform,
        fill=0,
        dtype="uint8",
    ).astype(bool)
    keep = valid & aoi_mask

    disp = np.zeros((th, EMB_RENDER_W, 3), dtype="float32")
    for i in range(3):
        lo, hi = np.percentile(cube[i][keep], (2, 98))
        disp[..., i] = np.clip((cube[i] - lo) / (hi - lo), 0.0, 1.0)
    rgba = np.dstack([disp, keep.astype("float32")])

    fig = plt.figure(figsize=(EMB_W_MM / 25.4, EMB_W_MM / t_aspect / 25.4), facecolor="none")
    ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(rgba, origin="upper", interpolation="nearest")
    ax.set_axis_off()
    ax.set_facecolor("none")
    # Keep the 600 dpi raster: it is already small, and at 300 dpi it would fall under
    # save_figure's 100k-pixel JPEG floor and be stored as (larger) lossless Flate instead.
    save_map(fig, name, dpi=RENDER_DPI)
    print(f"{name}: channels (R, G, B) {bands}")
    return fig


embedding_figure(TESSERA, f"{NOTEBOOK}/fig_1_panel_c.4_tessera_embedding")
plt.show()

## AlphaEarth embedding (false colour)

In [ ]:
embedding_figure(ALPHAEARTH, f"{NOTEBOOK}/fig_1_panel_c.3_alphaearth_embedding")
plt.show()